# `apptbl-flights-slopes`: Flights OLS treatment effects by attribute

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-flights-slopes"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
SRC_DIR = PAPER_DIR / "src"
if not (SRC_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from assets/; {SRC_DIR} has no microdata_analysis.py")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-flights-slopes.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [3]:
BETAS_FIELD = "kriner_betas_by_model"
FIRST_COLUMN_HEADER = "Flight attribute level"
HUMAN_HEADER = "Human"
MODEL_SHORT_LABELS = {"kev-0.8b": "0.8B", "kev-4b": "4B", "kev-9b": "9B"}
COMPARED_REGIMES = {BASE_SLUG: "Base", "row5_full_berke_wu_ce_oracle": "Pooled"}

# Optional renames, raw term -> text shown in the table. Terms not listed are shown as they are.
TERM_LABELS = {}

# Summary rows under the slopes: label -> which terms ("all", "price", "non-price"). {n} becomes the number of terms.
SUMMARY_ROWS = {r"$\mathrm{MSE}_\beta \times 10^3$": "all"}
PRICE_MARKER = "| Price:"

SLOPE_DECIMALS = 3
MSE_DECIMALS = 2
MSE_SCALE = 1000.0

## Rows (terms) in table order

In [4]:
# Terms starting with these prefixes come first, in this order (closest to Electricity's bill and renewables attributes).
LEADING_TERM_PREFIXES = ("Price:", "Fuel:")

raw_terms = list(getattr(ctx, BETAS_FIELD)[MODELS[0]][BASE_SLUG].index)
leading_terms = []
for prefix in LEADING_TERM_PREFIXES:
    prefix_terms = [term for term in raw_terms if term.startswith(prefix)]
    if not prefix_terms:
        raise ValueError(f"No Flights terms start with {prefix!r}: {raw_terms}")
    leading_terms.extend(prefix_terms)
other_terms = [term for term in raw_terms if term not in leading_terms]
terms = leading_terms + other_terms
print(f"{len(terms)} terms; {len(leading_terms)} leading")

12 terms; 4 leading


## Build the table

In [5]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table. column_rules set alignment (':---' left, '---:' right); dash counts come from column_widths_pt."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    widths_pt = column_widths_pt(headers, rows)
    rule_cells = []
    for column_rule, width_pt in zip(column_rules, widths_pt):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        dashes = "-" * round(width_pt)
        left = ":" if column_rule.startswith(":") else ""
        right = ":" if column_rule.endswith(":") else ""
        rule_cells.append(left + dashes + right)
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(rule_cells) + "|")
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [6]:
human_slopes = getattr(ctx, BETAS_FIELD)[MODELS[0]][BASE_SLUG].loc[terms, "human_pop_beta"].astype(float)

compared_columns = []
compared_slopes = []
for model_slug in MODELS:
    for regime_slug, regime_label in COMPARED_REGIMES.items():
        betas = getattr(ctx, BETAS_FIELD)[model_slug][regime_slug]
        compared_columns.append(f"{MODEL_SHORT_LABELS[model_slug]} {regime_label}")
        compared_slopes.append(betas.loc[terms, "model_pop_beta"].astype(float))

headers = [FIRST_COLUMN_HEADER, HUMAN_HEADER, *compared_columns]
rows = []
for term in terms:
    shown_term = TERM_LABELS[term] if term in TERM_LABELS else term
    row = [escape_markdown(shown_term), f"{human_slopes.loc[term]:.{SLOPE_DECIMALS}f}"]
    for slopes in compared_slopes:
        row.append(f"{slopes.loc[term]:.{SLOPE_DECIMALS}f}")
    rows.append(row)

for row_label_template, term_group in SUMMARY_ROWS.items():
    if term_group not in ("all", "price", "non-price"):
        raise ValueError(f"Unknown term group {term_group!r}")
    selected_terms = []
    for term in terms:
        is_price = PRICE_MARKER in term
        if term_group == "all":
            selected_terms.append(term)
        elif term_group == "price" and is_price:
            selected_terms.append(term)
        elif term_group == "non-price" and not is_price:
            selected_terms.append(term)
    if not selected_terms:
        raise ValueError(f"No terms selected for summary row {row_label_template!r}")
    row_label = row_label_template.replace("{n}", str(len(selected_terms)))
    row = [f"**{row_label}**", f"{0.0:.{MSE_DECIMALS}f}"]
    for slopes in compared_slopes:
        errors = slopes.loc[selected_terms] - human_slopes.loc[selected_terms]
        mse = float((errors ** 2).mean()) * MSE_SCALE
        row.append(f"**{mse:.{MSE_DECIMALS}f}**")
    rows.append(row)

column_rules = [":---"] + ["---:"] * (len(headers) - 1)
table_md = pipe_table(headers, rows, column_rules)
display(pd.DataFrame(rows, columns=headers))

,Flight attribute level,Human,0.8B Base,0.8B Pooled,4B Base,4B Pooled,9B Base,9B Pooled
0,Price: \$275,-0.085,0.003,-0.070,-0.055,-0.088,-0.027,-0.068
1,Price: \$300,-0.195,-0.013,-0.152,-0.101,-0.186,-0.064,-0.172
2,Price: \$325,-0.304,-0.020,-0.233,-0.164,-0.280,-0.127,-0.264
3,Fuel: Sustainable aviation fuel,0.069,-0.057,0.010,0.100,0.079,0.026,0.049
4,Airline: Frontier,-0.085,0.034,0.006,-0.180,-0.089,-0.127,-0.092
5,Airline: Southwest,-0.024,0.027,0.003,0.022,0.012,0.021,0.036
6,Airline: United,-0.022,0.001,0.002,-0.016,-0.010,-0.013,-0.012
7,Baggage: Checked bag+carry-on,0.056,-0.043,-0.026,0.087,0.069,0.086,0.089
8,Baggage: Personal item only,-0.139,-0.003,-0.014,-0.186,-0.092,-0.120,-0.103
9,Space: Extra legroom,0.065,-0.081,-0.003,0.124,0.114,0.043,0.088


## Save `apptbl-flights-slopes.md` (included by the manuscript)

In [7]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Flight attribute level | Human | 0.8B Base | 0.8B Pooled | 4B Base | 4B Pooled | 9B Base | 9B Pooled |
|:-----------------------------------------------------------------------------------------------------------------------------|------------------------------------------:|--------------------------------------------------:|---------------------------------------------------------:|---------------------------------------------:|----------------------------------------------------:|---------------------------------------------:|----------------------------------------------------:|
| Price: \$275 | -0.085 | 0.003 | -0.070 | -0.055 | -0.088 | -0.027 | -0.068 |
| Price: \$300 | -0.195 | -0.013 | -0.152 | -0.101 | -0.186 | -0.064 | -0.172 |
| Price: \$325 | -0.304 | -0.020 | -0.233 | -0.164 | -0.280 | -0.127 | -0.264 |
| Fuel: Sustainable aviation fuel | 0.069 | -0.057 | 0.010 | 0.100 | 0.079 | 0.026 | 0.049 |
| Airline: Frontier | -0.085 | 0.034 | 0.006 | -0.180 | -0.089 | -0.127 | -0.